<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/Megpie_RUL_Prediction_R03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.optimize import curve_fit
import datetime

# 한글 폰트 설정 (Windows: 'Malgun Gothic', Mac: 'AppleGothic')
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

# ==========================================
# 0. 가상 데이터(Dummy Data) 생성
# ==========================================
def generate_dummy_data():
    """웨이브, 노이즈, 이상치가 포함된 비선형 열화 데이터를 생성합니다."""
    np.random.seed(42)
    times = pd.date_range(start='2026-05-01', end='2026-05-30', freq='1h') # 1시간 단위
    n = len(times)

    # 기본 열화 베이스라인 (처음엔 0 유지하다가 중간 이후 지수적 증가)
    onset_idx = int(n * 0.5)
    base_val = np.zeros(n)
    base_val[onset_idx:] = np.exp(np.linspace(0, 3.5, n - onset_idx)) / np.exp(3.5) * 1.3

    # 노이즈 및 웨이브(사인파) 추가
    noise = np.random.normal(0, 0.03, n)
    wave = np.sin(np.linspace(0, 30, n)) * 0.08
    median_val = base_val + noise + wave

    # 이상치(Outlier) 추가
    median_val[int(n * 0.2)] = 0.9  # 튀는 값
    median_val[int(n * 0.7)] = 0.2  # 갑자기 떨어지는 값

    # 0 이하의 값은 0으로 클리핑 (보통 센서값이 음수가 되지 않는다고 가정)
    median_val = np.clip(median_val, 0, None)

    df = pd.DataFrame({'end_time': times, 'median_val': median_val})
    # 불규칙한 수집 주기 시뮬레이션: 데이터의 30%를 무작위로 삭제
    df = df.sample(frac=0.7, random_state=42).sort_values('end_time')
    return df

sel_df = generate_dummy_data()

# ==========================================
# 1. 데이터 전처리 (Preprocessing)
# ==========================================
df = sel_df.set_index('end_time').copy()

# 센서값 컬럼을 명시적으로 숫자형(float)으로 변환 (문자열 등이 섞여 있을 경우 NaN 처리 후 보간됨)
df['median_val'] = pd.to_numeric(df['median_val'], errors='coerce')

# [수정] 불규칙한 주기를 1시간 단위 정규 타임스탬프로 맞추고 결측치 선형 보간 (NaN, inf 방지)
# 문자열 형태의 다른 컬럼 때문에 나는 에러를 방지하기 위해 ['median_val'] 만 선택해서 리샘플링합니다.
df = df[['median_val']].resample('1h').mean()
df['median_val'] = df['median_val'].interpolate(method='time')
df['median_val'] = df['median_val'].bfill().ffill()

# 1-1. 이상치 완화를 위한 Moving Median (Window=5)
df['median_clean'] = df['median_val'].rolling(window=5, center=True).median()
df['median_clean'] = df['median_clean'].bfill().ffill() # 결측치 처리

# 1-2. Butterworth Low Pass Filter (노이즈 및 잔물결 제거)
def apply_lowpass_filter(data, cutoff_freq, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff_freq / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# 샘플링 주파수 fs=1(1시간에 1번), Cutoff: 24시간 주기의 변동 제거 (1/24)
df['smoothed'] = apply_lowpass_filter(df['median_clean'], cutoff_freq=1/24, fs=1)

# 1-3. 6시간 단위 리샘플링 (안전을 위해 최대값 Max 사용)
df_6h = df.resample('6h').max()
df_6h = df_6h.dropna()

# ==========================================
# 2. 열화 시작점 탐지 (Degradation Onset - Gradient Based)
# ==========================================
# [수정] 단순 임계값이 아닌, '상승 추세(기울기)'가 뚜렷한 구간 탐지
window = 3 # 18시간(3 * 6h) 전의 값과 비교하여 기울기 계산
df_6h['slope'] = (df_6h['smoothed'] - df_6h['smoothed'].shift(window)) / window

# 조건: 노이즈 플로어 이상이면서, 기울기가 일정 수준 이상 연속으로 지속되는 곳
noise_floor = 0.05
slope_threshold = 0.01
consecutive_points = 3

is_increasing = (df_6h['smoothed'] > noise_floor) & (df_6h['slope'] > slope_threshold)
onset_time = None
count = 0

for idx, val in is_increasing.items():
    if val:
        count += 1
        if count >= consecutive_points:
            # 상승이 본격적으로 시작된 시점(window 전)으로 거슬러 올라감
            onset_time = idx - pd.Timedelta(hours=6*(consecutive_points - 1 + window))
            # 인덱스 범위 보정
            onset_time = df_6h.index[df_6h.index >= onset_time][0] if onset_time < df_6h.index[-1] else idx
            break
    else:
        count = 0

# ==========================================
# 3. 모델 피팅 및 RUL 예측 (Extrapolation)
# ==========================================
failure_limit = 1.5
current_time = df_6h.index[-1]
predicted_failure_time = None

if onset_time is None:
    print("열화 시작점이 탐지되지 않았습니다. (정상 상태)")
else:
    # Onset 이후의 데이터만 추출하여 학습에 사용
    fit_data = df_6h[df_6h.index >= onset_time].copy()

    # [수정 1] 데이터 내의 잠재적인 inf/NaN 완벽 차단
    fit_data = fit_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['smoothed'])

    # 시간 데이터를 수치형(시간 단위)으로 변환 (fitting 안정성을 위함)
    x_data = (fit_data.index - onset_time).total_seconds() / 3600.0
    y_data = fit_data['smoothed'].values

    # [수정 2] 입력 데이터가 유효한지 다시 한번 필터링
    valid_mask = np.isfinite(x_data) & np.isfinite(y_data)
    x_data = x_data[valid_mask]
    y_data = y_data[valid_mask]

    if len(x_data) < 3:
        print("Not enough valid data points for curve fitting (minimum 3 required).")
    else:
        # 지수 함수 정의: y = a * exp(b * x) + c
        def exp_func(x, a, b, c):
            # [수정 3] 오버플로우로 인한 inf 반환을 막기 위해 철저히 클리핑
            val = a * np.exp(np.clip(b * x, -100, 100)) + c
            return np.clip(val, -1e10, 1e10)

        try:
            # [수정 4] 파라미터가 무한대로 발산하지 않도록 현실적인 상한선(Upper Bounds)과 초기값(p0) 설정
            # a, b는 0으로 나누는 에러 방지를 위해 최소값을 1e-8로 지정
            # a <= 50.0, b <= 1.0 (시간당 e^1 배 증가는 현실적으로 불가능한 속도), c <= 10.0
            initial_guess = (0.01, 0.001, np.min(y_data))
            popt, _ = curve_fit(
                exp_func,
                x_data,
                y_data,
                p0=initial_guess,
                bounds=([1e-8, 1e-8, -np.inf], [50.0, 1.0, 10.0]),
                maxfev=10000
            )
            a, b, c = popt

            # 예측: a * exp(b * t) + c = 1.5 가 되는 t를 계산
            if failure_limit > c:
                t_failure_hours = np.log((failure_limit - c) / a) / b
                predicted_failure_time = onset_time + pd.Timedelta(hours=t_failure_hours)
                rul = predicted_failure_time - current_time

                print(f"--- RUL Prediction Results ---")
                print(f"Current Time:      {current_time.strftime('%Y-%m-%d %H:%M')}")
                print(f"Degradation Onset: {onset_time.strftime('%Y-%m-%d %H:%M')}")
                print(f"Predicted Failure: {predicted_failure_time.strftime('%Y-%m-%d %H:%M')}")
                print(f"Remaining Useful Life (RUL): {rul.total_seconds() / 3600:.1f} hours ({rul.days} days {rul.components.hours} hours)")
            else:
                print("Cannot reach 1.5 due to formula structure.")

        except Exception as e:
            print(f"Curve Fitting Failed: {e}")

# ==========================================
# 4. 결과 시각화 (Visualization)
# ==========================================
plt.figure(figsize=(14, 7))

# 원본 및 전처리 데이터 플롯
plt.plot(df.index, df['median_val'], color='lightgray', label='Original Sensor Data (Irregular)', alpha=0.6)
plt.plot(df_6h.index, df_6h['smoothed'], color='blue', marker='o', markersize=4, label='Smoothed & Resampled Data (6H Max)')

# Upper Limit 선
plt.axhline(y=failure_limit, color='red', linestyle='--', linewidth=2, label=f'Upper Limit ({failure_limit})')

if onset_time is not None:
    # 열화 시작점 표시
    plt.axvline(x=onset_time, color='orange', linestyle=':', linewidth=2, label='Degradation Onset')
    if onset_time in df_6h.index:
        plt.plot(onset_time, df_6h.loc[onset_time, 'smoothed'], 'go', markersize=8, label='Onset Point')

    if predicted_failure_time is not None:
        # Extrapolation (예측 곡선) 그리기
        # 시작점부터 예상 고장 시점 + 약간의 여유 공간까지 시간축 생성
        future_times = pd.date_range(start=onset_time, end=predicted_failure_time + pd.Timedelta(days=2), freq='6h')
        future_x = (future_times - onset_time).total_seconds() / 3600.0
        future_y = exp_func(future_x, *popt)

        # 현재 시간 이후의 곡선만 점선으로 그리기 위해 필터링
        pred_mask = future_times >= current_time
        plt.plot(future_times[pred_mask], future_y[pred_mask], color='red', linestyle='-', linewidth=2, label='Predicted Trend (Extrapolation)')

        # 피팅에 사용된 구간은 실선으로 표시 (선택사항)
        fit_mask = future_times <= current_time
        plt.plot(future_times[fit_mask], future_y[fit_mask], color='green', linestyle='-', linewidth=2, label='Fitted Trend')

        # 고장 시점 마커
        plt.plot(predicted_failure_time, failure_limit, 'rX', markersize=10, label='Predicted Failure Point')

# 현재 시점 선
plt.axvline(x=current_time, color='black', linestyle='--', linewidth=1.5, label='Current Time')

plt.title("Megpie Component RUL Prediction (Low Pass Filter & Curve Fitting)", fontsize=16)
plt.xlabel("Time", fontsize=12)
plt.ylabel("Sensor Value", fontsize=12)
plt.ylim(0, 1.8)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()